# Triadic Cell Notebook v72
## Slot-Builder Training Corpus Generator

This notebook converts the recursive constructor line into a trainable corpus.

The target is:

$$
\boxed{Q \rightarrow C}
$$

where \(Q\) is the prompt and \(C\) is the missing-shape contract:

$$
C=(F,D,N,B,P,M,R,\Omega)
$$

- \(F\): family class
- \(D\): domain carrier
- \(N\): forbidden neighbor carrier
- \(B\): boundary conditions
- \(P\): preserved function
- \(M\): failure modes
- \(R\): witness/readout
- \(\Omega\): unresolved residue

This notebook reads whatever lineage outputs exist locally from v64-v71 and writes:

```text
slot_training_pairs.jsonl
slot_training_pairs.csv
slot_sft_messages.jsonl
slot_repair_messages.jsonl
slot_preference_pairs.jsonl
slot_corpus_manifest.json
```

Primary fine-tuning file:

```text
slot_sft_messages.jsonl
```

The model should learn to generate contracts, not answers.


In [11]:
from __future__ import annotations

import json, re
from pathlib import Path
from typing import Optional, List, Dict, Any

import numpy as np
import pandas as pd

OUTPUT_DIR = "v72_outputs_slot_builder_training_corpus"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

CANDIDATE_RUN_DIRS = [
    "v71_outputs_qwen25_1p5b_instruct_rotations_hurt_residue_backpatch_controller",
    "v70_outputs_qwen25_1p5b_instruct_rotations_dual_channel_recursive_controller",
    "v69_outputs_qwen25_1p5b_instruct_rotations_recursive_operational_checklist_controller",
    "v68_outputs_qwen25_1p5b_instruct_rotations_recursive_operational_checklist_critic",
    "v67_outputs_qwen25_1p5b_instruct_rotations_clean_recursive_slot_constructor",
    "v64_outputs_qwen25_1p5b_instruct_rotations_self_generating_slot_constructor",
]

RUN_ROOTS = [Path("."), Path("")]

def find_existing_run_dirs():
    found = []
    for root in RUN_ROOTS:
        for name in CANDIDATE_RUN_DIRS:
            p = root / name
            if p.exists() and p.is_dir() and p not in found:
                found.append(p)
        for p in root.glob("v*_outputs_*"):
            if p.is_dir() and p not in found:
                found.append(p)
    return found

run_dirs = find_existing_run_dirs()
print("Found run dirs:")
for p in run_dirs:
    print(" -", p)


Found run dirs:
 - v71_outputs_qwen25_1p5b_instruct_rotations_hurt_residue_backpatch_controller
 - v70_outputs_qwen25_1p5b_instruct_rotations_dual_channel_recursive_controller
 - v69_outputs_qwen25_1p5b_instruct_rotations_recursive_operational_checklist_controller
 - v68_outputs_qwen25_1p5b_instruct_rotations_recursive_operational_checklist_critic
 - v67_outputs_qwen25_1p5b_instruct_rotations_clean_recursive_slot_constructor
 - v64_outputs_qwen25_1p5b_instruct_rotations_self_generating_slot_constructor
 - v45_outputs_qwen25_1p5b_instruct
 - v46_outputs_qwen25_1p5b_instruct
 - v47_outputs_qwen25_1p5b_instruct
 - v48_outputs_qwen25_1p5b_instruct
 - v49_outputs_qwen25_1p5b_instruct
 - v50_outputs_qwen25_1p5b_instruct_rotations
 - v51_outputs_qwen25_1p5b_instruct_rotations
 - v53_outputs_qwen25_1p5b_instruct_rotations
 - v54_outputs_qwen25_1p5b_instruct_rotations_hybrid
 - v55_outputs_qwen25_1p5b_instruct_rotations_compiler_root
 - v56_outputs_qwen25_1p5b_instruct_rotations_abstract_slot
 

In [12]:
# -----------------------------
# File loading helpers
# -----------------------------
def read_csv_if_exists(path: Path) -> Optional[pd.DataFrame]:
    if path.exists():
        try:
            return pd.read_csv(path)
        except Exception as e:
            print("Could not read", path, e)
    return None

def write_jsonl(path: Path, rows: List[dict]):
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def split_pipe_field(x):
    if x is None:
        return []
    try:
        if pd.isna(x):
            return []
    except Exception:
        pass
    if isinstance(x, list):
        return x
    s = str(x)
    return [p.strip() for p in s.split("|") if p.strip()]

def normalize_contract_from_row(row: dict) -> dict:
    return {
        "family_class": str(row.get("family_class", "") or "").strip(),
        "domain_carrier": split_pipe_field(row.get("domain_carrier", "")),
        "forbidden_neighbor_carrier": split_pipe_field(row.get("forbidden_neighbor_carrier", "")),
        "boundary_conditions": split_pipe_field(row.get("boundary_conditions", "")),
        "preserved_function": str(row.get("preserved_function", "") or "").strip(),
        "failure_modes": split_pipe_field(row.get("failure_modes", "")),
        "witness_readout": str(row.get("witness_readout", "") or "").strip(),
        "residue": None if row.get("residue", None) is None or str(row.get("residue", "")).lower() in ["nan", "none", "null", ""] else row.get("residue", None),
    }

def compact_contract(contract: dict) -> dict:
    keys = [
        "family_class",
        "domain_carrier",
        "forbidden_neighbor_carrier",
        "boundary_conditions",
        "preserved_function",
        "failure_modes",
        "witness_readout",
        "residue",
    ]
    return {k: contract.get(k, None) for k in keys}

def operational_audit_from_row(row: dict) -> dict:
    def f(k):
        try:
            return float(row.get(k, np.nan))
        except Exception:
            return np.nan
    return {
        "quality": f("op_quality") if "op_quality" in row else f("quality"),
        "F_need": f("F_need"),
        "F_function": f("F_function"),
        "F_boundary": f("F_boundary"),
        "F_trap": f("F_trap"),
        "F_collapse": f("F_collapse"),
        "failures": split_pipe_field(row.get("op_failures", row.get("failures", ""))),
    }

print("Helpers loaded.")


Helpers loaded.


In [13]:
# -----------------------------
# Load lineage artifacts
# -----------------------------
artifacts = []

for run_dir in run_dirs:
    patch_dir = run_dir / "v71_hurt_residue_patches"

    artifact = {
        "run_dir": run_dir,
        "summary": read_csv_if_exists(run_dir / "summary.csv"),
        "results": read_csv_if_exists(run_dir / "results.csv"),
        "interesting": read_csv_if_exists(run_dir / "interesting_cases.csv"),
        "initial_contracts": read_csv_if_exists(run_dir / "initial_contracts.csv"),
        "repaired_contracts": read_csv_if_exists(run_dir / "repaired_contracts.csv"),
        "operational_repaired_contracts": read_csv_if_exists(run_dir / "operational_repaired_contracts.csv"),
        "operational_audit": read_csv_if_exists(run_dir / "operational_repaired_audit.csv"),
        "operational_trace": read_csv_if_exists(run_dir / "operational_repair_trace.csv"),
        "v71_patch_audit": read_csv_if_exists(run_dir / "v71_patch_audit.csv"),
        "v71_best_eval": read_csv_if_exists(run_dir / "v71_best_eval_rows.csv"),
        "v71_best_fail": read_csv_if_exists(run_dir / "v71_best_fail_rows.csv"),
        "v71_hurt_records": read_csv_if_exists(patch_dir / "hurt_records_from_best_explore.csv"),
        "v71_patched_contracts": read_csv_if_exists(patch_dir / "patched_contracts.csv"),
    }
    artifacts.append(artifact)

manifest_rows = []
for a in artifacts:
    for k, v in a.items():
        if k == "run_dir":
            continue
        manifest_rows.append({
            "run_dir": str(a["run_dir"]),
            "artifact": k,
            "found": v is not None,
            "rows": 0 if v is None else len(v),
        })

manifest_df = pd.DataFrame(manifest_rows)
display(manifest_df[manifest_df.found].sort_values(["run_dir", "artifact"]))


Could not read v71_outputs_qwen25_1p5b_instruct_rotations_hurt_residue_backpatch_controller\v71_hurt_residue_patches\hurt_records_from_best_explore.csv No columns to parse from file


,run_dir,artifact,found,rows
79,v45_outputs_qwen25_1p5b_instruct,results,True,4
93,v46_outputs_qwen25_1p5b_instruct,interesting,True,3
92,v46_outputs_qwen25_1p5b_instruct,results,True,26
106,v47_outputs_qwen25_1p5b_instruct,interesting,True,2
105,v47_outputs_qwen25_1p5b_instruct,results,True,26
...,...,...,...,...
0,v71_outputs_qwen25_1p5b_instruct_rotations_hur...,summary,True,1
9,v71_outputs_qwen25_1p5b_instruct_rotations_hur...,v71_best_eval,True,96
10,v71_outputs_qwen25_1p5b_instruct_rotations_hur...,v71_best_fail,True,89
8,v71_outputs_qwen25_1p5b_instruct_rotations_hur...,v71_patch_audit,True,24


In [14]:
# -----------------------------
# Select best available artifacts
# -----------------------------
def version_num(p):
    m = re.search(r"v(\d+)_outputs", str(p))
    return int(m.group(1)) if m else -1

def first_nonempty(key):
    ordered = sorted(artifacts, key=lambda a: version_num(a["run_dir"]), reverse=True)
    for a in ordered:
        df = a.get(key)
        if df is not None and len(df):
            return a["run_dir"], df
    return None, None

source_results_dir, results_df = first_nonempty("results")
source_initial_dir, initial_contracts_df = first_nonempty("initial_contracts")
source_operational_dir, operational_contracts_df = first_nonempty("operational_repaired_contracts")
source_repaired_dir, repaired_contracts_df = first_nonempty("repaired_contracts")
source_patched_dir, patched_contracts_df = first_nonempty("v71_patched_contracts")
source_hurt_dir, hurt_records_df = first_nonempty("v71_hurt_records")
source_patch_audit_dir, patch_audit_df = first_nonempty("v71_patch_audit")

print("Selected sources:")
print("results:", source_results_dir, None if results_df is None else len(results_df))
print("initial contracts:", source_initial_dir, None if initial_contracts_df is None else len(initial_contracts_df))
print("operational contracts:", source_operational_dir, None if operational_contracts_df is None else len(operational_contracts_df))
print("repaired contracts:", source_repaired_dir, None if repaired_contracts_df is None else len(repaired_contracts_df))
print("patched contracts:", source_patched_dir, None if patched_contracts_df is None else len(patched_contracts_df))
print("hurt records:", source_hurt_dir, None if hurt_records_df is None else len(hurt_records_df))
print("patch audit:", source_patch_audit_dir, None if patch_audit_df is None else len(patch_audit_df))

if results_df is None:
    raise RuntimeError("No results.csv found. Run v71/v68/v67 first, then rerun v72.")


Selected sources:
results: v71_outputs_qwen25_1p5b_instruct_rotations_hurt_residue_backpatch_controller 96
initial contracts: v71_outputs_qwen25_1p5b_instruct_rotations_hurt_residue_backpatch_controller 24
operational contracts: v71_outputs_qwen25_1p5b_instruct_rotations_hurt_residue_backpatch_controller 24
repaired contracts: v65_outputs_qwen25_1p5b_instruct_rotations_contract_critic_repair_gate 24
patched contracts: v71_outputs_qwen25_1p5b_instruct_rotations_hurt_residue_backpatch_controller 24
hurt records: None None
patch audit: v71_outputs_qwen25_1p5b_instruct_rotations_hurt_residue_backpatch_controller 24


## Corpus construction logic

v72 creates three training objects:

### 1. SFT rows

$$
Q \rightarrow C_{\text{target}}
$$

The model sees the prompt and emits the best available contract.

Target priority:

1. v71 patched contract
2. v68 operational repaired contract
3. v67 repaired contract
4. v64 initial/generated contract

### 2. Repair rows

$$
(Q,C_0,A_0,\Omega) \rightarrow C_1
$$

The model learns to repair a weak or unsafe contract.

### 3. Preference rows

$$
(Q,C_{\text{chosen}},C_{\text{rejected}})
$$

Chosen = patched/repaired contract.  
Rejected = initial contract.


In [15]:
# -----------------------------
# Build contract maps
# -----------------------------
def df_to_contract_map(df: Optional[pd.DataFrame], id_col="base_id"):
    m = {}
    if df is None:
        return m
    for _, r in df.iterrows():
        if id_col not in r or pd.isna(r[id_col]):
            continue
        bid = str(r[id_col])
        m[bid] = normalize_contract_from_row(dict(r))
    return m

initial_map = df_to_contract_map(initial_contracts_df)
operational_map = df_to_contract_map(operational_contracts_df)
repaired_map = df_to_contract_map(repaired_contracts_df)
patched_map = df_to_contract_map(patched_contracts_df)

target_contract_map = {}
all_ids = set()
for m in [initial_map, operational_map, repaired_map, patched_map]:
    all_ids.update(m.keys())

for bid in sorted(all_ids):
    if bid in patched_map:
        target_contract_map[bid] = patched_map[bid]
    elif bid in operational_map:
        target_contract_map[bid] = operational_map[bid]
    elif bid in repaired_map:
        target_contract_map[bid] = repaired_map[bid]
    elif bid in initial_map:
        target_contract_map[bid] = initial_map[bid]

print("contract maps:")
print("initial:", len(initial_map))
print("operational:", len(operational_map))
print("repaired:", len(repaired_map))
print("patched:", len(patched_map))
print("target:", len(target_contract_map))


contract maps:
initial: 24
operational: 24
repaired: 24
patched: 24
target: 24


In [16]:
# -----------------------------
# Prompt map and result map
# -----------------------------
prompt_map = {}

if "prompt" in results_df.columns:
    for _, r in results_df.iterrows():
        bid = str(r.get("base_id", ""))
        if bid and bid not in prompt_map:
            prompt_map[bid] = str(r.get("prompt", ""))
else:
    for _, r in results_df.iterrows():
        bid = str(r.get("base_id", ""))
        if bid and bid not in prompt_map:
            prompt_map[bid] = (
                "Generate the missing-shape contract for this Nexus task. "
                f"Gold choice observed: {r.get('gold_choice', '')}"
            )

for df in [initial_contracts_df, operational_contracts_df, repaired_contracts_df, patched_contracts_df]:
    if df is not None and "prompt" in df.columns:
        for _, r in df.iterrows():
            bid = str(r.get("base_id", ""))
            if bid and bid not in prompt_map:
                prompt_map[bid] = str(r.get("prompt", ""))

result_map = {}
if results_df is not None and "base_id" in results_df.columns:
    group_cols = ["base_correct", "compiled_correct", "generated_correct", "generated_helped", "generated_hurt", "generated_omega"]
    existing = [c for c in group_cols if c in results_df.columns]
    if existing:
        g = results_df.groupby("base_id")[existing].mean().reset_index()
        for _, r in g.iterrows():
            result_map[str(r["base_id"])] = dict(r)

print("prompt map:", len(prompt_map))
print("result map:", len(result_map))


prompt map: 24
result map: 24


In [17]:
# -----------------------------
# Hurt residue map and audit map
# -----------------------------
hurt_map = {}
if hurt_records_df is not None and len(hurt_records_df):
    for bid, grp in hurt_records_df.groupby("base_id"):
        hurt_map[str(bid)] = grp.to_dict(orient="records")

audit_map = {}

if patch_audit_df is not None and len(patch_audit_df) and "base_id" in patch_audit_df.columns:
    for _, r in patch_audit_df.iterrows():
        audit_map[str(r["base_id"])] = {
            "quality": float(r.get("quality", np.nan)),
            "F_need": float(r.get("F_need", np.nan)),
            "F_function": float(r.get("F_function", np.nan)),
            "F_boundary": float(r.get("F_boundary", np.nan)),
            "F_trap": float(r.get("F_trap", np.nan)),
            "F_collapse": float(r.get("F_collapse", np.nan)),
            "failures": split_pipe_field(r.get("failures", "")),
        }

if results_df is not None and "base_id" in results_df.columns:
    op_cols = ["op_quality", "F_need", "F_function", "F_boundary", "F_trap", "F_collapse", "op_failures"]
    if any(c in results_df.columns for c in op_cols):
        g = results_df.groupby("base_id").first().reset_index()
        for _, r in g.iterrows():
            bid = str(r["base_id"])
            if bid not in audit_map:
                audit_map[bid] = {
                    "quality": float(r.get("op_quality", np.nan)) if "op_quality" in r else np.nan,
                    "F_need": float(r.get("F_need", np.nan)) if "F_need" in r else np.nan,
                    "F_function": float(r.get("F_function", np.nan)) if "F_function" in r else np.nan,
                    "F_boundary": float(r.get("F_boundary", np.nan)) if "F_boundary" in r else np.nan,
                    "F_trap": float(r.get("F_trap", np.nan)) if "F_trap" in r else np.nan,
                    "F_collapse": float(r.get("F_collapse", np.nan)) if "F_collapse" in r else np.nan,
                    "failures": split_pipe_field(r.get("op_failures", "")),
                }

print("hurt map:", len(hurt_map))
print("audit map:", len(audit_map))


hurt map: 0
audit map: 24


In [18]:
# -----------------------------
# Prompt formatting
# -----------------------------
SYSTEM_SLOT_BUILDER = """You are the Nexus Slot Constructor.

Your job is to generate the missing-shape contract before answer selection.

Do not answer the task.
Do not mention answer choices.
Return strict JSON only.

The contract must contain:
family_class
domain_carrier
forbidden_neighbor_carrier
boundary_conditions
preserved_function
failure_modes
witness_readout
residue

Use operational fit, not labels.
"""

def make_user_prompt_for_sft(prompt: str) -> str:
    return f"""Prompt:
{prompt}

Generate the missing-shape contract.

Checklist:
1. Need: occupy the inverse cavity.
2. Function: preserve or redirect the required operation.
3. Boundary: respect constraints.
4. Trap: reject noun/surface-label confusion.
5. Collapse: produce one executable witness/readout.

Return JSON only."""

def make_user_prompt_for_repair(prompt: str, initial_contract: dict, audit: dict, hurt_residue: list) -> str:
    return f"""Prompt:
{prompt}

Current contract:
{json.dumps(compact_contract(initial_contract), ensure_ascii=False, indent=2)}

Operational audit:
{json.dumps(audit, ensure_ascii=False, indent=2)}

Hurt / Ω residue:
{json.dumps(hurt_residue, ensure_ascii=False, indent=2)}

Repair the contract only.
Do not answer the task.
Return strict JSON only."""

def to_messages(system: str, user: str, assistant_obj: dict) -> dict:
    return {
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
            {"role": "assistant", "content": json.dumps(compact_contract(assistant_obj), ensure_ascii=False, indent=2)},
        ]
    }

print("Prompt templates loaded.")


Prompt templates loaded.


In [19]:
# -----------------------------
# Create corpus rows
# -----------------------------
slot_training_pairs = []
slot_sft_messages = []
slot_repair_messages = []
slot_preference_pairs = []

for bid, target_contract in sorted(target_contract_map.items()):
    prompt = prompt_map.get(bid, "")
    if not prompt:
        prompt = f"Generate the missing-shape contract for Nexus task {bid}."

    initial_contract = initial_map.get(bid, {})
    operational_contract = operational_map.get(bid, {})
    repaired_contract = repaired_map.get(bid, {})
    patched_contract = patched_map.get(bid, {})
    audit = audit_map.get(bid, {})
    hurt_residue = hurt_map.get(bid, [])
    result = result_map.get(bid, {})

    source_priority = (
        "patched" if bid in patched_map else
        "operational_repaired" if bid in operational_map else
        "repaired" if bid in repaired_map else
        "initial"
    )

    row = {
        "base_id": bid,
        "prompt": prompt,
        "initial_contract": compact_contract(initial_contract) if initial_contract else None,
        "operational_contract": compact_contract(operational_contract) if operational_contract else None,
        "repaired_contract": compact_contract(repaired_contract) if repaired_contract else None,
        "patched_contract": compact_contract(patched_contract) if patched_contract else None,
        "target_contract": compact_contract(target_contract),
        "target_source": source_priority,
        "operational_audit": audit,
        "hurt_residue": hurt_residue,
        "result": result,
    }
    slot_training_pairs.append(row)

    sft = to_messages(SYSTEM_SLOT_BUILDER, make_user_prompt_for_sft(prompt), target_contract)
    sft["base_id"] = bid
    sft["target_source"] = source_priority
    slot_sft_messages.append(sft)

    if initial_contract and target_contract:
        repair = to_messages(
            SYSTEM_SLOT_BUILDER,
            make_user_prompt_for_repair(prompt, initial_contract, audit, hurt_residue),
            target_contract,
        )
        repair["base_id"] = bid
        repair["target_source"] = source_priority
        slot_repair_messages.append(repair)

        if compact_contract(initial_contract) != compact_contract(target_contract):
            slot_preference_pairs.append({
                "base_id": bid,
                "prompt": make_user_prompt_for_repair(prompt, initial_contract, audit, hurt_residue),
                "chosen": json.dumps(compact_contract(target_contract), ensure_ascii=False, indent=2),
                "rejected": json.dumps(compact_contract(initial_contract), ensure_ascii=False, indent=2),
                "metadata": {
                    "target_source": source_priority,
                    "has_hurt_residue": bool(hurt_residue),
                    "audit": audit,
                    "result": result,
                }
            })

print("slot_training_pairs:", len(slot_training_pairs))
print("slot_sft_messages:", len(slot_sft_messages))
print("slot_repair_messages:", len(slot_repair_messages))
print("slot_preference_pairs:", len(slot_preference_pairs))


slot_training_pairs: 24
slot_sft_messages: 24
slot_repair_messages: 24
slot_preference_pairs: 24


In [20]:
# -----------------------------
# Save corpus
# -----------------------------
out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)

write_jsonl(out / "slot_training_pairs.jsonl", slot_training_pairs)
write_jsonl(out / "slot_sft_messages.jsonl", slot_sft_messages)
write_jsonl(out / "slot_repair_messages.jsonl", slot_repair_messages)
write_jsonl(out / "slot_preference_pairs.jsonl", slot_preference_pairs)

csv_rows = []
for r in slot_training_pairs:
    tc = r["target_contract"]
    audit = r.get("operational_audit", {}) or {}
    result = r.get("result", {}) or {}
    csv_rows.append({
        "base_id": r["base_id"],
        "target_source": r["target_source"],
        "prompt": r["prompt"],
        "family_class": tc.get("family_class", ""),
        "domain_carrier": " | ".join(tc.get("domain_carrier", []) or []),
        "forbidden_neighbor_carrier": " | ".join(tc.get("forbidden_neighbor_carrier", []) or []),
        "boundary_conditions": " | ".join(tc.get("boundary_conditions", []) or []),
        "preserved_function": tc.get("preserved_function", ""),
        "failure_modes": " | ".join(tc.get("failure_modes", []) or []),
        "witness_readout": tc.get("witness_readout", ""),
        "residue": tc.get("residue", None),
        "quality": audit.get("quality", np.nan),
        "F_need": audit.get("F_need", np.nan),
        "F_function": audit.get("F_function", np.nan),
        "F_boundary": audit.get("F_boundary", np.nan),
        "F_trap": audit.get("F_trap", np.nan),
        "F_collapse": audit.get("F_collapse", np.nan),
        "has_hurt_residue": bool(r.get("hurt_residue")),
        "generated_correct": result.get("generated_correct", np.nan),
        "generated_hurt": result.get("generated_hurt", np.nan),
        "generated_helped": result.get("generated_helped", np.nan),
    })

corpus_df = pd.DataFrame(csv_rows)
corpus_df.to_csv(out / "slot_training_pairs.csv", index=False)

manifest = {
    "output_dir": str(out),
    "n_slot_training_pairs": len(slot_training_pairs),
    "n_slot_sft_messages": len(slot_sft_messages),
    "n_slot_repair_messages": len(slot_repair_messages),
    "n_slot_preference_pairs": len(slot_preference_pairs),
    "source_results_dir": str(source_results_dir) if source_results_dir else None,
    "source_initial_dir": str(source_initial_dir) if source_initial_dir else None,
    "source_operational_dir": str(source_operational_dir) if source_operational_dir else None,
    "source_patched_dir": str(source_patched_dir) if source_patched_dir else None,
    "source_hurt_dir": str(source_hurt_dir) if source_hurt_dir else None,
    "files": {
        "slot_training_pairs_jsonl": "slot_training_pairs.jsonl",
        "slot_training_pairs_csv": "slot_training_pairs.csv",
        "slot_sft_messages_jsonl": "slot_sft_messages.jsonl",
        "slot_repair_messages_jsonl": "slot_repair_messages.jsonl",
        "slot_preference_pairs_jsonl": "slot_preference_pairs.jsonl",
    }
}
(out / "slot_corpus_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

display(corpus_df.head(20))
print(json.dumps(manifest, indent=2))


,base_id,target_source,prompt,family_class,domain_carrier,forbidden_neighbor_carrier,boundary_conditions,preserved_function,failure_modes,witness_readout,...,quality,F_need,F_function,F_boundary,F_trap,F_collapse,has_hurt_residue,generated_correct,generated_hurt,generated_helped
0,adv_api_01,patched,Generate the missing-shape contract for this N...,missing-shape,api | call | exposes | one | method | hiding |...,authentication | routing | validation | persis...,"{'valid': 'method exposed', 'invalid': ['hidde...",operational event,incorrect method exposure | unauthorized acces...,an API call exposing one method while hiding a...,...,0.761812,0.895085,0.788926,0.785057,0.859922,0.649961,False,1.00,0.0,0.0
1,adv_breath_01,patched,Generate the missing-shape contract for this N...,operational closure of breathes non-moving mat...,non-moving | matter | changes,moving matter | surface label without operatio...,"{'preserve': 'without', 'reject': 'moving'}",functionality of non-moving matter preservation,collapse due to moving nature | failure in mai...,the chosen collapse preserves non-moving matte...,...,0.600772,0.713159,0.625001,0.764216,0.827176,0.529184,False,0.50,0.0,0.0
2,adv_car_01,patched,Generate the missing-shape contract for this N...,interface-collapse,car | hides | combustion | gearing | sensors |...,combustion | surface label without operational...,"{'preserve': ['gearing', 'sensors', 'tire fric...",combustion,general operation family | data processing | a...,"a car hiding combustion, gearing, sensors, tir...",...,0.744425,0.882501,0.641829,0.833335,0.748665,0.772342,False,1.00,0.0,0.0
3,adv_commit_01,patched,Generate the missing-shape contract for this N...,operational closure of possible repair real un...,potential | commitment path | readout,general operation family | operation family,"preserve: commitment path | reject: collapse, ...","repair potential after collapse, ensuring comm...","surface label mismatch in collapse, readout fa...","the chosen collapse preserves possible, repair...",...,0.697196,0.825528,0.825105,0.834610,0.704632,0.605022,False,1.00,0.0,0.0
4,adv_constraint_01,patched,Generate the missing-shape contract for this N...,fold handling,handles | doing | crease | fold line,rotate | surface label without operational fit...,"{'preserve': 'the original shape', 'reject': '...",maintaining the integrity of the folded structure,discontinuity at the fold line | tearing along...,a visible crease or fold line on the surface,...,0.680585,0.678336,0.618189,0.890622,0.770928,0.597052,False,0.50,0.0,0.0
5,adv_coupler_01,patched,Generate the missing-shape contract for this N...,spinning,"rubber coupler, vacuum pump shaft",concrete | metal | surface label without opera...,"{'valid': ['radial compression'], 'invalid': [...",rotation transmission,coupler loosening | shaft damage | pump malfun...,observed radial compression without coupler mo...,...,0.726387,0.668946,0.697019,0.858375,0.799621,0.700505,False,0.00,0.0,0.0
6,adv_coupler_02,patched,Generate the missing-shape contract for this N...,compressive coupling,motion | missing | noun | rubber | part | cent...,flexible joint | surface label without operati...,{'preserve': 'centered compressive coupling un...,"preserve the operation implied by missing, nou...",inability to maintain centered compressive cou...,the system exhibits consistent behavior during...,...,0.736162,0.757077,0.667084,0.798890,0.783015,0.770632,False,1.00,0.0,0.0
7,adv_flower_01,patched,Generate the missing-shape contract for this N...,operational closure of flower described hidden...,flower | hidden | rather | color,visible | bloom | describes | surface color,"{'preserve': ['hidden', 'rather'], 'reject': [...",remaining as a flower described hidden rather ...,disappearing or becoming invisible | surface w...,"the chosen collapse preserves flower, hidden, ...",...,0.668965,0.728250,0.822190,0.820869,0.763678,0.555549,False,1.00,0.0,0.0
8,adv_fold_01,patched,Generate the missing-shape contract f

{
  "output_dir": "v72_outputs_slot_builder_training_corpus",
  "n_slot_training_pairs": 24,
  "n_slot_sft_messages": 24,
  "n_slot_repair_messages": 24,
  "n_slot_preference_pairs": 24,
  "source_results_dir": "v71_outputs_qwen25_1p5b_instruct_rotations_hurt_residue_backpatch_controller",
  "source_initial_dir": "v71_outputs_qwen25_1p5b_instruct_rotations_hurt_residue_backpatch_controller",
  "source_operational_dir": "v71_outputs_qwen25_1p5b_instruct_rotations_hurt_residue_backpatch_controller",
  "source_patched_dir": "v71_outputs_qwen25_1p5b_instruct_rotations_hurt_residue_backpatch_controller",
  "source_hurt_dir": null,
  "files": {
    "slot_training_pairs_jsonl": "slot_training_pairs.jsonl",
    "slot_training_pairs_csv": "slot_training_pairs.csv",
    "slot_sft_messages_jsonl": "slot_sft_messages.jsonl",
    "slot_repair_messages_jsonl": "slot_repair_messages.jsonl",
    "slot_preference_pairs_jsonl": "slot_preference_pairs.jsonl"
  }
}


## Optional QLoRA training sketch

The corpus target is:

```text
slot_sft_messages.jsonl
```

The trained behavior is:

$$
Q \rightarrow C
$$

not:

$$
Q \rightarrow \text{answer}
$$

Pseudo-command:

```bash
python train_slot_builder_lora.py \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --train_file v72_outputs_slot_builder_training_corpus/slot_sft_messages.jsonl \
  --output_dir slot_builder_lora_v1 \
  --max_seq_length 2048 \
  --load_in_4bit true \
  --lora_r 16 \
  --lora_alpha 32 \
  --target_modules q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj
```

After training:

```text
Base model + Slot-builder LoRA
  ↓
generated missing-shape contract
  ↓
operational critic
  ↓
KRRB resolver
  ↓
hurt residue
  ↓
next corpus
```

That is manifold grooving:

$$
\text{error}
\rightarrow
\text{residue}
\rightarrow
\text{constraint}
\rightarrow
\text{training memory}
\rightarrow
\text{instinct}
$$
